# Phase 1: Data Preprocessing, Copying, and Dataset Pipelines

In this phase, you will write the code to:
1. Copy the split image files from `data/raw` into structured directories under `data/processed/`.
2. Load these processed directories into TensorFlow `tf.data.Dataset` objects.
3. Implement data augmentation and dataset performance optimization (caching & prefetching).

### Checkpoint 1.1: Copy split files to `data/processed`

Write a function or loop that iterates through your split paths (`train_paths`, `val_paths`, `test_paths`) and copies the files to their respective destination in `data/processed/`.

**Expected Directory Structure:**
```text
data/processed/
├── train/
│   ├── Normal/
│   ├── Bacterial/
│   └── Viral/
├── val/
│   ├── Normal/
│   ├── Bacterial/
│   └── Viral/
└── test/
    ├── Normal/
    ├── Bacterial/
    └── Viral/
```

*Hint:* Use `os.makedirs(..., exist_ok=True)` to create the directories, and `shutil.copy(src, dst)` to copy files.

In [ ]:
# Write your code here to copy files from raw splits to data/processed splits
import os
import shutil
from sklearn.model_selection import train_test_split

# 1. Define paths to where your raw data sits
raw_dir = '../data/raw/'
categories = ['NORMAL', 'PNEUMONIA']

all_filepaths = []
all_labels = []

# 2. Walk through raw folders and parse true multi-class labels from names
for folder in ['train', 'test', 'val']:
    for category in categories:
        folder_path = os.path.join(raw_dir, folder, category)
        if not os.path.exists(folder_path):
            continue
            
        for filename in os.listdir(folder_path):
            if filename.endswith('.jpeg') or filename.endswith('.png'):
                full_path = os.path.join(folder_path, filename)
                
                # Determine class based on folder and filename pattern
                if category == 'NORMAL':
                    label = 'Normal'
                elif 'bacteria' in filename.lower():
                    label = 'Bacterial'
                elif 'virus' in filename.lower():
                    label = 'Viral'
                else:
                    continue # Skip anomalies
                
                all_filepaths.append(full_path)
                all_labels.append(label)

# 3. Create your proper Stratified Splits (70% Train, 15% Val, 15% Test)
# This completely fixes the "16-image validation folder" issue from Kaggle
train_paths, test_paths, train_labels, test_labels = train_test_split(
    all_filepaths, all_labels, test_size=0.30, stratify=all_labels, random_state=42
)

val_paths, test_paths, val_labels, test_labels = train_test_split(
    test_paths, test_labels, test_size=0.50, stratify=test_labels, random_state=42
)

print(f"Train size: {len(train_paths)} | Val size: {len(val_paths)} | Test size: {len(test_paths)}")

processed_dir = '../data/processed'

# Define mapping for splits
splits = {
    'train': (train_paths, train_labels),
    'val': (val_paths, val_labels),
    'test': (test_paths, test_labels)
}

# Loop through each split and copy the files
for split_name, (paths, labels) in splits.items():
    print(f"Copying {split_name} split...")
    for path, label in zip(paths, labels):
        # Construct the target directory
        dest_dir = os.path.join(processed_dir, split_name, label)
        
        # Create the directory if it doesn't exist
        os.makedirs(dest_dir, exist_ok=True)
        
        # Copy the image from its raw path to the target directory
        shutil.copy(path, dest_dir)

print("All files copied successfully!")


Train size: 4099 | Val size: 878 | Test size: 879


### Checkpoint 1.2: Load directories into TensorFlow Datasets

Use `tf.keras.utils.image_dataset_from_directory` to load the `train`, `val`, and `test` subfolders from `data/processed/`.

Set the following parameters:
- `image_size=(224, 224)`
- `batch_size=32`
- `label_mode='categorical'` (to output one-hot encoded labels for our 3 classes)

In [ ]:
import tensorflow as tf

# Load train, val, and test datasets from directory
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
data_dir = '../'
# train_ds = tf.keras.utils.image_dataset_from_directory(...)
# val_ds = ...
# test_ds = ...


### Checkpoint 1.3: Define Data Augmentation layers

To fight overfitting, build a data augmentation pipeline using Keras preprocessing layers. Combine `RandomFlip`, `RandomRotation`, and `RandomZoom` layers into a `tf.keras.Sequential` block.

In [ ]:
# Create data augmentation sequential layers
# data_augmentation = tf.keras.Sequential([
#     tf.keras.layers.RandomFlip(...),
#     ...
# ])


### Checkpoint 1.4: Configure Dataset caching and prefetching

Apply `.cache()` and `.prefetch(buffer_size=tf.data.AUTOTUNE)` to your loaded datasets to optimize performance during training.

In [ ]:
# Optimize dataset loaders for performance
AUTOTUNE = tf.data.AUTOTUNE

# train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
# val_ds = ...
# test_ds = ...
